# Rehaussement de contraste : une méthode par paradigme

Pour chaque méthode : image, histogramme (gris = original, bleu = résultat) et valeurs clés. Tableau récapitulatif à la fin.
Les implémentations sont dans `contraste.py`.

**Valeurs clés** — `contraste_rms` : écart-type · `dynamique` : percentiles 1–99 · `entropie` : bits (max 8) · `nettete` : gradient moyen · `bruit` : écart-type estimé · `nettete/bruit` : en hausse = les bords gagnent plus que le bruit.

In [ ]:
%load_ext autoreload
%autoreload 2
import cv2
import pandas as pd
import contraste as ct

IMAGE = "resultats/Run1-1I1RA-ACTI-PH-0A_Target-I175100-ACTI-PH-DLH-0A_FX1_FY-2_RR1_TR1_Ver1_sans_annotations.png"
gris = cv2.imread(IMAGE, cv2.IMREAD_GRAYSCALE)
R = {"Original": ct.mesures(gris)}

## 1. Transformations globales

**Étirement linéaire** — Les percentiles 1 et 99 sont envoyés sur 0 et 255.

In [ ]:
R["Étirement linéaire"] = ct.comparer(gris, ct.etirement(gris), "Étirement linéaire")

**Gamma** — $I^{\gamma}$ : $\gamma<1$ éclaircit les sombres, $\gamma>1$ l'inverse.

In [ ]:
R["Gamma"] = ct.comparer(gris, ct.gamma(gris, 0.6), "Gamma")

**Égalisation** — Courbe = histogramme cumulé : l'histogramme de sortie devient à peu près plat.

In [ ]:
R["Égalisation"] = ct.comparer(gris, ct.egalisation(gris), "Égalisation")

## 2. Histogramme local

**CLAHE** — Égalisation par tuiles, histogramme écrêté (`clip`) pour limiter l'amplification du bruit.

In [ ]:
R["CLAHE"] = ct.comparer(gris, ct.clahe(gris, clip=2.0, tuiles=8), "CLAHE")

## 3. Éclairage × réflectance

**Homomorphique** — log → passe-haut gaussien dans Fourier (atténue le fond lent, renforce les détails) → exp.

In [ ]:
R["Homomorphique"] = ct.comparer(gris, ct.homomorphique(gris, sigma=20, gain_bas=0.5, gain_haut=1.5), "Homomorphique")

**Retinex multi-échelle** — Éclairage estimé par flous gaussiens à 3 échelles, divisé (en log).

In [ ]:
R["Retinex multi-échelle"] = ct.comparer(gris, ct.retinex(gris, sigmas=(15, 80, 250)), "Retinex multi-échelle")

## 4. Base + détails

**Filtre guidé** — Base = filtre guidé (lisse sans flouter les bords), détails = image − base, amplifiés par `gain`.

In [ ]:
R["Filtre guidé"] = ct.comparer(gris, ct.details_guide(gris, rayon=8, eps=0.01, gain=3), "Filtre guidé")

## 5. Multi-échelle

**Pyramide laplacienne** — Chaque niveau de détail (fin → grossier) est multiplié par un gain, puis on reconstruit.

In [ ]:
R["Pyramide laplacienne"] = ct.comparer(gris, ct.pyramide_laplacienne(gris, gains=(1.2, 2, 2, 1.5)), "Pyramide laplacienne")

**Ondelettes** — Même idée sur les coefficients de détail d'une décomposition en ondelettes (db2).

In [ ]:
R["Ondelettes"] = ct.comparer(gris, ct.ondelettes(gris, gains=(1.2, 2, 2)), "Ondelettes")

## 6. Morphologie mathématique

**Top-hat** — $I$ + top-hat blanc − top-hat noir : les structures plus petites que `rayon` ressortent.

In [ ]:
R["Top-hat"] = ct.comparer(gris, ct.tophat(gris, rayon=6), "Top-hat")

## 7. Optimisation (domaine du gradient)

**Poisson** — Gradients remodelés ($\beta<1$ renforce les faibles), puis image minimisant $\|\nabla u - G\|^2$.

In [ ]:
R["Poisson"] = ct.comparer(gris, ct.gradient_poisson(gris, beta=0.8), "Poisson")

## 8. Fusion d'expositions

**Mertens** — Trois versions gamma fusionnées : chaque pixel prend les versions bien exposées et contrastées.

In [ ]:
R["Mertens"] = ct.comparer(gris, ct.fusion_expositions(gris, gammas=(0.5, 1, 2)), "Mertens")

## 9. Logique floue

**Intensification floue** — Niveaux → appartenance μ ∈ [0, 1], poussée vers 0 ou 1 autour de 0,5 (Pal et King).

In [ ]:
R["Intensification floue"] = ct.comparer(gris, ct.floue(gris, iterations=1), "Intensification floue")

## 10. Apprentissage profond (non supervisé)

**Zero-DCE** — Petit CNN qui prédit une courbe de correction par pixel, entraîné sur l'image seule sans référence (~40 s sur CPU). Conçu pour les images sous-exposées : il ramène surtout la luminance vers 0,5.

In [ ]:
R["Zero-DCE"] = ct.comparer(gris, ct.zero_dce(gris, iterations=100), "Zero-DCE")

## Récapitulatif
Trié par `nettete/bruit` ; écarts relatifs à l'original en %.

In [ ]:
T = pd.DataFrame(R).T
ecart = (100 * (T / T.loc["Original"] - 1)).add_suffix(" %")
T.join(ecart[["contraste_rms %", "nettete/bruit %"]]).sort_values("nettete/bruit", ascending=False).round(2)

## Pistes pour préparer l'entrée de SAM

Trois idées : **débruiter avant de contraster**, **régler le CLAHE plutôt que le prendre par défaut**, **rendre les images homogènes entre elles**.

**NLM → CLAHE** — Le CLAHE amplifie aussi le grain MEB ; un NLM juste avant le prive de bruit à amplifier.

In [ ]:
R["NLM → CLAHE"] = ct.comparer(gris, ct.nlm_clahe(gris), "NLM → CLAHE")

**Réglage du CLAHE** — `clip` = force (plus haut : plus de contraste et de bruit) ; `tuiles` = taille de la zone locale (plus de tuiles : plus local). Sans puis avec NLM avant.

In [ ]:
ct.grille_clahe(gris)

In [ ]:
ct.grille_clahe(gris, debruiter=True)

**Diffusion anisotrope (Perona-Malik)** — Lissage qui s'arrête aux bords (gradient > `kappa`) : débruite sans arrondir les contours, alternative au NLM.

In [ ]:
R["Perona-Malik"] = ct.comparer(gris, ct.perona_malik(gris, iterations=15, kappa=15), "Perona-Malik")

In [ ]:
R["Perona-Malik → CLAHE"] = ct.comparer(gris, ct.clahe(ct.perona_malik(gris)), "Perona-Malik → CLAHE")

**Spécification d'histogramme** — Donne à une image l'histogramme d'une image de référence. Pour SAM, surtout utile pour un fine-tuning : toutes les images (entraînement comme test) arrivent avec la même distribution de gris, quelle que soit l'exposition.

In [ ]:
REFERENCE = "resultats/Run1-1I1RA-LIN3-PH-0A_Target-I175100-LIN3-PH-DSH-0A_M1_FX0_FY0_RR1_TR1_Ver1_sans_annotations.png"
reference = cv2.imread(REFERENCE, cv2.IMREAD_GRAYSCALE)
R["Spécification"] = ct.comparer(gris, ct.specification(gris, reference), "Spécification")

**Composite RGB** — SAM attend 3 canaux et reçoit d'ordinaire 3 fois la même image grise. On peut y mettre 3 vues : brute (R), NLM (G), NLM + CLAHE (B). Effet incertain en zero-shot (SAM a appris sur des photos en couleur), plus intéressant avec un fine-tuning.

In [ ]:
import matplotlib.pyplot as plt
rgb = ct.composite_rgb(gris)
fig, ax = plt.subplots(1, 4, figsize=(14, 3.5))
for a, img, t in zip(ax, [rgb, *rgb.transpose(2, 0, 1)], ["Composite", "R : brute", "G : NLM", "B : NLM + CLAHE"]):
    a.imshow(img, cmap=None if img.ndim == 3 else "gray", vmin=0, vmax=255); a.set_title(t, fontsize=10); a.axis("off")
plt.show()

## Récapitulatif (avec les pistes)

In [ ]:
T = pd.DataFrame(R).T
T.sort_values("nettete/bruit", ascending=False).round(2)

## Effet sur SAM

Le contraste « pour l'œil » ne dit pas ce que SAM en fera : on compare donc directement ses masques (grille de points, SAM 2.1 small, ~2-3 min).
`stabilite` = deux points voisins du même objet donnent-ils le même masque ? `iou_verite` n'apparaît que si l'image a des annotations dans `segmentation/` : c'est la seule mesure décisive.

In [ ]:
import importlib, segmentation_sam, segsem
for m in (segmentation_sam, segsem, ct):
    importlib.reload(m)

import os
import numpy as np
import segmentation as seg

nom = os.path.splitext(os.path.basename(IMAGE))[0]
chemin_labels = f"segmentation/{nom}_labels.png"
labels = cv2.imread(chemin_labels, cv2.IMREAD_UNCHANGED).astype(np.int32) if os.path.exists(chemin_labels) else None

ct.effet_sam(gris, {
    "Original": lambda g: g,
    "Étirement": ct.etirement,
    "CLAHE": ct.clahe,
    "Ondelettes": ct.ondelettes,
    "Pyramide laplacienne": ct.pyramide_laplacienne,
    "NLM": ct.nlm,
    "NLM + étirement": lambda g: ct.etirement(ct.nlm(g)),
    "NLM + ondelettes": lambda g: ct.ondelettes(ct.nlm(g)),
    "NLM → CLAHE": ct.nlm_clahe,
    "Perona-Malik → CLAHE": lambda g: ct.clahe(ct.perona_malik(g)),
    "Composite RGB": ct.composite_rgb,
}, labels)

**Voir les masques** — Segmentation automatique par SAM seul (grille de points) pour chaque prétraitement, superposée à l'image d'origine. Sont écartés les masques de plus de la moitié de l'image (SAM prend souvent tout le fond d'un bloc) et ceux de score < `score_min` (fragments de fond). ~10 s par variante.

In [ ]:
import importlib, segmentation_sam, segsem
for m in (segmentation_sam, segsem, ct):
    importlib.reload(m)

ct.voir_sam(gris, {
    "Original": lambda g: g,
    "CLAHE": ct.clahe,
    "NLM → CLAHE": ct.nlm_clahe,
    "NLM + étirement": lambda g: ct.etirement(ct.nlm(g)),
    "Composite RGB": ct.composite_rgb,
    "Perona-Malik → CLAHE": lambda g: ct.clahe(ct.perona_malik(g)),
})

## Comparer les tailles de SAM

Même prétraitement, SAM 2.1 tiny → large : masques côte à côte, temps d'encodage, score moyen, et `iou_verite` si l'image est annotée.
Premier lancement : téléchargement des modèles manquants (base+ ~320 Mo, large ~900 Mo). Compter ~1 à 3 min par modèle sur CPU.

In [ ]:
import importlib, segmentation_sam, segsem
for m in (segmentation_sam, segsem, ct):
    importlib.reload(m)

ct.comparer_modeles(gris, ct.nlm_clahe, labels)

## Invites données par une méthode classique

Le watershed trouve les régions, SAM ne fait qu'affiner leur contour : une invite par région (point le plus intérieur, puis boîte englobante + point). Une boîte lève l'ambiguïté « objet ou tout le fond ? » qu'un point seul laisse. ~30 s.

In [ ]:
import importlib, segmentation_sam, segsem
for m in (segmentation_sam, segsem, ct):
    importlib.reload(m)

ct.comparer_invites(gris, ct.nlm_clahe, labels)

## Contours précis (sous-pixel et zoom)

SAM prédit ses masques en 256 × 256 : sur une image de 480 px, un pixel de masque couvre ~2 px, d'où des bords en escalier. Deux remèdes comparés sur un objet :
**sous-pixel** = ligne de niveau 0 des logits (interpolée) au lieu du masque seuillé ; **zoom** = SAM n'encode que la boîte de l'objet, agrandie à 1024 px.
Trait pointillé blanc : ligne où la largeur est mesurée. `objet=` choisit une autre région du watershed.

In [ ]:
import importlib, segmentation_sam, segsem
for m in (segmentation_sam, segsem, ct):
    importlib.reload(m)

ct.contours_precis(gris, ct.nlm_clahe, labels=labels)